### Objetivo

##### Utilizar técnicas de clasificación aprendidas hasta el momento para predecir la calidad del vino basándose en características físico-químicas. Este ejercicio permitirá aplicar conceptos como la selección de características, preprocesamiento de datos, entrenamiento y evaluación de modelos de clasificación, y análisis de resultados mediante métricas y visualizaciones.

##### Descripción del Dataset: Este conjunto de datos contiene información sobre distintas características físico-químicas de muestras de vino tinto y su calidad asociada. Las características incluyen acidez fija, acidez volátil, ácido cítrico, azúcar residual, cloruros, dióxido de azufre libre, dióxido de azufre total, densidad, pH, sulfatos y alcohol. La calidad del vino está clasificada en una escala del 0 al 10.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV , cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

#### 1. Carga y Exploración de Datos:



- 1.1 Cargar el dataset y revisar su estructura básica.


In [4]:
# 1.1 Cargar el dataset
df = pd.read_csv('data/WineQT.csv')

print("\ndimensiones")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}\n")

print("\nprimeras 5 filas\n")
display(df.head(10))

# los ultimos 5 registros
display("\nultimos 5 registros : \n",df.tail())


print("\n nombres de las columnas \n")
display(df.columns.tolist())

print("\ntipos de datos \n")
print(df.info())

print("\n estadisticas descriptivas \n")
display(df.describe())


dimensiones
Filas: 1143, Columnas: 13


primeras 5 filas



,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,5
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5,6
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7,7
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7,8
9,6.7,0.58,0.08,1.8,0.097,15.0,65.0,0.9959,3.28,0.54,9.2,5,10


'\nultimos 5 registros : \n'

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
1138,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,1592
1139,6.8,0.620,0.08,1.9,0.068,28.0,38.0,0.99651,3.42,0.82,9.5,6,1593
1140,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,1594
1141,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,1595
1142,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,1597



 nombres de las columnas 



['fixed acidity',
 'volatile acidity',
 'citric acid',
 'residual sugar',
 'chlorides',
 'free sulfur dioxide',
 'total sulfur dioxide',
 'density',
 'pH',
 'sulphates',
 'alcohol',
 'quality',
 'Id']


tipos de datos 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB
None

 estadisticas descriptivas 



,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000
mean,8.311111,0.531339,0.268364,2.532152,0.086933,15.615486,45.914698,0.996730,3.311015,0.657708,10.442111,5.657043,804.969379
std,1.747595,0.179633,0.196686,1.355917,0.047267,10.250486,32.782130,0.001925,0.156664,0.170399,1.082196,0.805824,463.997116
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000,0.000000
25%,7.100000,0.392500,0.090000,1.900000,0.070000,7.000000,21.000000,0.995570,3.205000,0.550000,9.500000,5.000000,411.000000
50%,7.900000,0.520000,0.250000,2.200000,0.079000,13.000000,37.000000,0.996680,3.310000,0.620000,10.200000,6.000000,794.000000
75%,9.100000,0.640000,0.420000,2.600000,0.090000,21.000000,61.000000,0.997845,3.400000,0.730000,11.100000,6.000000,1209.500000
max,15.900000,1.580000,1.000000,15.500000,0.611000,68.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000,1597.000000


- 1.2 Describir las variables y su distribución.


|	Columna	|	Tipo	|	Descripcion	|
|	 ---------------------	|	 ------------	|	 -----------------------	|
|'	fixed acidity	 '|	float64	|	Acidez fija	|
|'	volatile acidity	 '|	float64	|	Acidez volatil	|
|'	citric acid	 '|	float64	|	Acido citrico	|
|'	residual sugar	 '|	float64	|	Azucar residual	|
|'	chlorides	 '|	float64	|	Cloruros	|
|'	free sulfur dioxide	 '|	float64	|	Dioxido de azufre libre	|
|'	total sulfur dioxide	 '|	float64	|	Dioxido de azufre total	|
|'	density	 '|	float64	|	Densidad	|
|'	pH	 '|	float64	|	pH	|
|'	sulphates	 '|	float64	|	Sulfatos	|
|'	alcohol	 '|	float64	|	Grado alcoholico	|
|'	quality	 '|	int64	|	Variable objetivo (3-8)	|
|'	Id	 '|	int64	|	Identificador	|


<!-- |	Columna	|	Tipo	|	Descripcion	|
|	 ---------------------	|	 ------------	|	 -----------------------	|
|'	fixed acidity	 '|	float64	|	Acidez fija	|
|'	volatile acidity	 '|	float64	|	Acidez volatil	|
|'	citric acid	 '|	float64	|	Acido citrico	|
|'	residual sugar	 '|	float64	|	Azucar residual	|
|'	chlorides	 '|	float64	|	Cloruros	|
|'	free sulfur dioxide	 '|	float64	|	Dioxido de azufre libre	|
|'	total sulfur dioxide	 '|	float64	|	Dioxido de azufre total	|
|'	density	 '|	float64	|	Densidad	|
|'	pH	 '|	float64	|	pH	|
|'	sulphates	 '|	float64	|	Sulfatos	|
|'	alcohol	 '|	float64	|	Grado alcoholico	|
|'	quality	 '|	int64	|	Variable objetivo (3-8)	|
|'	Id	 '|	int64	|	Identificador	| -->


- 1.3 Identificar y tratar valores nulos y outliers.

In [6]:
# verificar datos nulos en el dataframe
print("\nDatos nulos por columna : \n")
print(df.isnull().sum())
print("\nDatos nulos totales : ", df.isnull().sum().sum())

print("\nNo se encuentran valores nulos en el dataset, por lo que no es necesario realizar imputación de datos.")



Datos nulos por columna : 

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64

Datos nulos totales :  0

No se encuentran valores nulos en el dataset, por lo que no es necesario realizar imputación de datos.


In [7]:
# deteccion de outliarns en el dataframe
print("\n deteccion de outliarns en el dataframe.\n")
# columnas numericas
colum_numericas = [col for col in df if df[col].dtype in ['int64', 'float64']]

print("\n columnas numericas : ", colum_numericas)

# columnas categoricas
colum_categoricas = [col for col in df if df[col].dtype == 'object']

print("\n columnas categoricas :  ", colum_categoricas, "\n")

def deteccion_outliers(df_e, column):
    Q1 = df_e[column].quantile(0.25)
    Q3 = df_e[column].quantile(0.75)
    IQR = Q3 - Q1
    limit_inf = Q1 - 1.5 * IQR
    limit_sup = Q3 + 1.5 * IQR
    outliers = df_e[(df_e[column] < limit_inf) | (df_e[column] > limit_sup)]
    return outliers, limit_inf, limit_sup

def remover_outliers(df_e, column):
    Q1 = df_e[column].quantile(0.25)
    Q3 = df_e[column].quantile(0.75)
    IQR = Q3 - Q1
    limit_inf = Q1 - 1.5 * IQR
    limit_sup = Q3 + 1.5 * IQR
    df_out = df_e[(df_e[column] >= limit_inf) & (df_e[column] <= limit_sup)]
    return df_out

def mostrar_correccion_outliers(df_lim, column, arr_outliers):
   # print("arr_outliers", arr_outliers[column])

    existe = any(arr== column for arr in arr_outliers)
    #print("existe : ", existe)
    if existe: 
        outliers = df_lim[(df_lim[column] < arr_outliers[column]['lower']) | (df_lim[column] > arr_outliers[column]['upper'])]
 #       print(f"outliers {df_lim[column]} " )
 #       display("mostrar outliers", outliers)
        count = len(outliers) # cuenta la cantidad de outliers encontrados en la columna
        print(f"{column:18} -> cantidad de outliers = {count:3} | limites : [{arr_outliers[column]['lower']:,.2f}, {arr_outliers[column]['upper']:,.2f}]")

arr_outliers = {}

for col in colum_numericas:
        outliers, lower, upper = deteccion_outliers(df, col)
        # Guardas los valores

        count = len(outliers) # cuenta la cantidad de outliers encontrados en la columna
        if count > 0:
            print(f"{col:18} -> cantidad de outliers = {count:3} | limites : [{lower:,.2f}, {upper:,.2f}]")
            nombre_outlier = col
            arr_outliers.update({nombre_outlier: {'upper': upper, 'lower': lower}})


 deteccion de outliarns en el dataframe.


 columnas numericas :  ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'Id']

 columnas categoricas :   [] 

fixed acidity      -> cantidad de outliers =  44 | limites : [4.10, 12.10]
volatile acidity   -> cantidad de outliers =  14 | limites : [0.02, 1.01]
citric acid        -> cantidad de outliers =   1 | limites : [-0.40, 0.91]
residual sugar     -> cantidad de outliers = 110 | limites : [0.85, 3.65]
chlorides          -> cantidad de outliers =  77 | limites : [0.04, 0.12]
free sulfur dioxide -> cantidad de outliers =  18 | limites : [-14.00, 42.00]
total sulfur dioxide -> cantidad de outliers =  40 | limites : [-39.00, 121.00]
density            -> cantidad de outliers =  36 | limites : [0.99, 1.00]
pH                 -> cantidad de outliers =  20 | limites : [2.91, 3.69]
sulphates          -> cantidad de 

In [ ]:
print

for col in colum_numericas:            
    df_clean = remover_outliers(df_clean, col)

print("\n mostrar corrección de outlierns. \n ")
#display('df_limpio',df_limpio['length'])
#display(arr_outliers)
for col in colum_numericas:
     #print(f"columna : {col} ")
     mostrar_correccion_outliers(df_clean, col, arr_outliers)

